# Notebook Ingesta Completa

**Fuente de Datos:** Archivos Excel (Marcobre Fileserver)  
**Destino:** Delta Lake (Unity Catalog)  
**Modo de Actualización:** Sobrescritura Completa (Patrón Full Copy)

## Descripción General
Este notebook realiza la ingesta completa de archivos Excel en tablas Delta Lake. Soporta orientaciones de datos tanto verticales como horizontales, encabezados multinivel, mapeo personalizado de columnas y extracción de metadatos (fechas de reporte y nombres de pronóstico). El notebook lee desde la zona pre-landing Bronze y escribe en la zona landing Bronze con evolución de esquema habilitada.

## Requisitos Previos
- **Notebook de Funciones**: Requiere el notebook `../Funciones/Ingesta Funciones` con funciones utilitarias (`parse_excel_range_advanced`, `apply_datetime_tokens`, `build_headers`, `sanitize_column`, `fix_duplicate_columns`, `normalize_dataframe_types`, `drop_all_nan_rows`, `slugify`)
- **Widgets**: Todos los parámetros deben proporcionarse mediante widgets del notebook (ExcelPath, TableName, SheetName, DataRange, etc.)
- **Volúmenes**: Acceso a los volúmenes de Unity Catalog `{environment}_bronze_prelanding_minsur` y `{environment}_bronze_landing_minsur`
- **Librerías**: `pandas`, `openpyxl`, `delta-spark`

## 1. Importar Funciones Compartidas
Cargar funciones utilitarias comunes desde el notebook Functions (parse_excel_range_advanced, apply_datetime_tokens, sanitize_column, etc.)

In [0]:
%run "../Funciones/Ingesta Funciones"

## 2. Importar Librerías y Configurar Spark
Importar librerías Python requeridas y configurar ajustes de Spark para el procesamiento de Excel

In [0]:
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import current_timestamp

# Deshabilitar optimización Arrow para evitar problemas de compatibilidad Spark-Pandas durante la conversión de DataFrame
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

# Suprimir advertencias de openpyxl sobre formato de archivos Excel que no afectan la lectura de datos
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

## 3. Definir Widgets del Notebook (Parámetros)
Crear widgets interactivos para aceptar parámetros de configuración para el proceso de ingestión.
Estos valores serán pasados desde el pipeline de orquestación.

In [0]:
# Identificador de entorno (dev, test, prod) usado para la construcción de rutas de volúmenes
dbutils.widgets.text("Environment", "", "Environment")

# Ruta del archivo Excel fuente (soporta tokens de datetime como {yyyyMMddHH})
dbutils.widgets.text("ExcelPath", "", "Excel Path")

# Nombre de la tabla Delta destino en el catálogo
dbutils.widgets.text("TableName", "", "Table Name")

# Nombre específico de la hoja dentro del libro de Excel
dbutils.widgets.text("SheetName", "", "Sheet Name")

# Rango de celdas que define los límites de datos (ej. "A1:Z100")
dbutils.widgets.text("DataRange", "", "Data Range")

# Índices de columnas a extraer cuando se usa mapeo personalizado de columnas (separados por coma)
dbutils.widgets.text("IndexColumns", "", "Index Columns")

# Nombres de columnas personalizados a asignar cuando se usa extracción basada en índices (separados por coma)
dbutils.widgets.text("CustomColumnNames", "", "Custom Column Names")

# Referencia de celda que contiene el valor de fecha de reporte (ej. "B2")
dbutils.widgets.text("ReportDate", "", "Report Date")

# Referencia de celda que contiene el identificador de pronóstico (ej. "C1")
dbutils.widgets.text("ForecastName", "", "Forecast Name")

# Rangos de columnas a omitir durante la lectura (ej. "B:D,F")
dbutils.widgets.text("ColumnsToSkip", "", "Columns To Skip")

# Rangos de filas a omitir durante la lectura (ej. "1:3,5")
dbutils.widgets.text("RowsToSkip", "", "Rows To Skip")

# Bandera para indicar si los datos están orientados horizontalmente (requiere transposición)
dbutils.widgets.text("IsHorizontal", "", "Is Horizontal")

# Número de filas de encabezado a fusionar para encabezados Excel multinivel
dbutils.widgets.text("MergeNHeaders", "", "Merge N Headers")

# Marca de tiempo de ejecución para reemplazo de tokens en la ruta de archivo (por defecto hora UTC actual)
dbutils.widgets.text("Datetime", "", "Datetime")

## 4. Recuperar Valores de Widgets y Construir Rutas
Extraer valores de parámetros de los widgets y construir la ruta final de archivo usando reemplazo de tokens de datetime

In [0]:
# Recuperar valores de configuración de los widgets del notebook
excel_file_path = dbutils.widgets.get("ExcelPath")
table_name = dbutils.widgets.get("TableName")
excel_sheet_name = dbutils.widgets.get("SheetName")
excel_cell_range = dbutils.widgets.get("DataRange")
index_columns = dbutils.widgets.get("IndexColumns")
custom_column_names = dbutils.widgets.get("CustomColumnNames")
report_date = dbutils.widgets.get("ReportDate")
forecast_name = dbutils.widgets.get("ForecastName")
skip_col_range = dbutils.widgets.get("ColumnsToSkip")
skip_row_range = dbutils.widgets.get("RowsToSkip")

# Convertir booleano en cadena a booleano real para la bandera de orientación
is_horizontal = (True if str(dbutils.widgets.get("IsHorizontal")).lower() == "true" else False)

# Analizar número de filas de encabezado a fusionar (por defecto 1 si no se especifica)
header_rows = int(dbutils.widgets.get("MergeNHeaders") or 1)

# Obtener datetime de ejecución o usar por defecto hora UTC actual
current_datetime = str(dbutils.widgets.get("Datetime") or datetime.now("UTC").strftime("%Y-%m-%d %H:%M:%S"))

# Obtener identificador de entorno para construcción de rutas de volúmenes
environment = dbutils.widgets.get("Environment")

# Aplicar reemplazo de tokens de datetime a la ruta de archivo (ej. {yyyyMMddHH} -> 2024012414)
final_path = apply_datetime_tokens(excel_file_path, datetime.strptime(current_datetime, "%Y-%m-%d %H:%M:%S"))

# Construir ruta completa de volumen DBFS usando volumen externo de Unity Catalog
input_excel_path = f"/Volumes/{environment}_bronze_prelanding_minsur/marcobre_fileserver/marcobre_fileserver/{final_path}"

# Salida de depuración para verificar construcción de rutas y ajustes de orientación
print(f"Input path: {input_excel_path}")
print(f"Is horizontal: {is_horizontal}")
print(f"Header rows to merge: {header_rows}")

## 5. Analizar Configuración de Rango de Excel
Convertir especificaciones de rango legibles por humanos en parámetros compatibles con pandas

In [0]:
# Analizar la cadena de rango de Excel y configuraciones de omisión en parámetros estructurados
# Retorna diccionario con: usecols, skiprows, nrows para pandas.read_excel()
params = parse_excel_range_advanced(
    table_range=excel_cell_range,
    skip_col_range=skip_col_range,
    skip_row_range=skip_row_range
)

# Salida de depuración para verificar lógica de análisis de rango
print("Parsed Excel parameters:")
print(params)

## 6. Leer Datos de Excel
Cargar datos desde archivo Excel con soporte para orientaciones tanto horizontal (transpuesta) como vertical

In [0]:
# Leer datos crudos de Excel sin procesamiento de encabezados (header=None)
# Manejaremos la construcción de encabezados manualmente para soportar encabezados multifila
df = pd.read_excel(
    input_excel_path,
    sheet_name=excel_sheet_name,
    usecols=params['usecols'],
    skiprows=params['skiprows'],
    nrows=params['nrows'],
    header=None,  # Deshabilitar detección automática de encabezados
    engine="openpyxl"
)

# Verificar si los datos están orientados horizontalmente (requieren rotación de 90 grados)
if is_horizontal:
    # Transponer el DataFrame para convertir diseño horizontal a vertical
    df_t = df.T.reset_index(drop=True)
    
    # Verificar si se especifica mapeo personalizado de columnas (IndexColumns + CustomColumnNames)
    if index_columns != "" and custom_column_names != "":
        # Analizar índices de columnas y nombres personalizados de cadenas separadas por coma
        col_indexes = [int(i) for i in index_columns.split(",")]
        new_column_names = [c.strip() for c in custom_column_names.split(",")]
        
        # Validar que el conteo de índices coincida con el conteo de nombres
        if len(col_indexes) != len(new_column_names):
            raise ValueError(
                f"Number of index columns ({len(col_indexes)}) does not match "
                f"number of custom column names ({len(new_column_names)})"
            )
        
        # Extraer solo columnas especificadas y asignar nombres personalizados
        df_final = df_t.iloc[:, col_indexes]
        df_final.columns = new_column_names
    else:
        # Construir encabezados compuestos desde primeras N filas usando la función utilitaria build_headers
        headers = build_headers(df_t, header_rows)
        df_t.columns = headers
        
        # Remover las filas de encabezado de los datos reales
        df_final = df_t.iloc[header_rows:].reset_index(drop=True)
else:
    # Orientación vertical (diseño estándar de Excel)
    if index_columns != "" and custom_column_names != "":
        # Analizar índices de columnas (conversión de base-1 a base-0)
        col_indexes = [int(i) - 1 for i in index_columns.split(",")]
        new_column_names = [c.strip() for c in custom_column_names.split(",")]
        
        # Validar configuración de mapeo
        if len(col_indexes) != len(new_column_names):
            raise ValueError(
                f"Number of index columns ({len(col_indexes)}) does not match "
                f"number of custom column names ({len(new_column_names)})"
            )
        
        # Extraer columnas especificadas con nomenclatura personalizada
        df_final = df.iloc[:, col_indexes]
        df_final.columns = new_column_names
    else:
        # Construir encabezados desde primeras N filas para soporte de encabezado multinivel
        headers = build_headers(df, header_rows)
        df.columns = headers
        
        # Remover filas de encabezado de datos y reiniciar índice
        df = df.iloc[header_rows:].reset_index(drop=True)
        df_final = df.copy()

## 7. Extraer Campos de Metadatos
Opcionalmente extraer fecha de reporte y nombre de pronóstico desde celdas específicas en el archivo Excel

In [0]:
# Inicializar contenedores de valores de metadatos
report_date_val = ""
forecast_name_val = ""

# Extraer fecha de reporte si se proporciona referencia de celda
if report_date.strip():
    # Analizar la referencia de celda (ej. "B2" -> usecols y skiprows)
    report_params = parse_excel_range_advanced(f"{report_date.strip()}:{report_date.strip()}")
    
    # Leer la celda única que contiene la fecha de reporte
    report_df = pd.read_excel(
        input_excel_path,
        sheet_name=excel_sheet_name,
        usecols=report_params['usecols'],
        skiprows=report_params['skiprows'],
        nrows=report_params['nrows'],
        header=None,
        engine="openpyxl"
    )
    
    # Extraer el valor escalar del DataFrame de celda única
    report_date_val = str(report_df.iloc[0, 0])
    
    # Agregar columna report_date al conjunto de datos final
    df_final = df_final.copy()
    df_final.loc[:, "report_date"] = report_date_val

# Extraer nombre de pronóstico si se proporciona referencia de celda
if forecast_name.strip():
    # Analizar la referencia de celda para nombre de pronóstico
    forecast_params = parse_excel_range_advanced(f"{forecast_name.strip()}:{forecast_name.strip()}")
    
    # Leer la celda única que contiene el identificador de pronóstico
    forecast_df = pd.read_excel(
        input_excel_path,
        sheet_name=excel_sheet_name,
        usecols=forecast_params['usecols'],
        skiprows=forecast_params['skiprows'],
        nrows=forecast_params['nrows'],
        header=None,
        engine="openpyxl"
    )
    
    # Extraer valor de nombre de pronóstico
    forecast_name_val = str(forecast_df.iloc[0, 0])
    
    # Agregar columna forecast al conjunto de datos final
    df_final = df_final.copy()
    df_final.loc[:, "forecast"] = forecast_name_val

## 8. Limpieza y Transformación de Datos
Sanitizar nombres de columnas, manejar duplicados, normalizar tipos de datos y remover filas vacías

In [0]:
# Almacenar nombres de columnas originales para seguimiento de transformación y depuración
original_columns = df_final.columns.tolist()

# Sanitizar nombres de columnas: minúsculas, reemplazar caracteres especiales con guiones bajos, normalizar acentos
df_final.columns = [sanitize_column(c) for c in df_final.columns]

# Resolver nombres de columnas duplicados agregando sufijos numéricos (col, col_1, col_2, etc.)
df_final.columns = fix_duplicate_columns(df_final.columns)

# Normalizar tipos de datos: convertir cadenas a numéricos donde sea posible, estandarizar fechas, etc.
df_final = normalize_dataframe_types(df_final)

# Remover filas donde todos los valores son NaN (filas completamente vacías)
df_final = drop_all_nan_rows(df_final)

# Preparar sufijo de nombre de pronóstico para nomenclatura de tabla (si aplica)
final_forecast_name = ""
if forecast_name_val:
    final_forecast_name = f"_{forecast_name_val.lower().replace(' ', '_')}"

# Mostrar mapeo de transformación de nombres de columnas para propósitos de auditoría
print("Column mapping (Original -> Sanitized):")
for original, sanitized in zip(original_columns, df_final.columns):
    print(f"  {original}  -->  {sanitized}")

## 9. Convertir a Spark DataFrame y Vista Previa
Convertir el DataFrame pandas limpio a Spark DataFrame para operaciones de Delta Lake

In [0]:
# Convertir DataFrame pandas a Spark DataFrame para procesamiento distribuido
df_table = spark.createDataFrame(df_final)

# Mostrar vista previa de los datos (primeras 1000 filas) para validación
display(df_table)

## 10. Escribir en Delta Lake
Guardar los datos procesados como tabla Delta en la capa Bronze con evolución de esquema habilitada

In [0]:
# Normalizar nombre de hoja para nomenclatura de tabla (remover espacios, puntos, barras)
final_excel_sheet = excel_sheet_name.lower().replace(' ', '_').replace('.', '').replace('/', '_')

# Normalizar nombre de tabla usando utilidad slugify para asegurar identificador válido
final_destination_table = f"{slugify(table_name.lower().replace(' ', '_').replace('.', '').replace('/', '_'))}"
new_table_name = f"{final_destination_table}"

# Escribir Spark DataFrame a tabla Delta en Unity Catalog
# - format("delta"): Usar formato Delta Lake para transacciones ACID
# - option("mergeSchema", "true"): Habilitar evolución de esquema para agregar nuevas columnas automáticamente
# - mode("overwrite"): Reemplazar contenido de tabla existente (patrón de copia completa)
df_table = df_table.withColumn("ingested_at", current_timestamp())

df_table.write \
  .format("delta") \
  .option("mergeSchema", "true") \
  .mode("append") \
  .saveAsTable(f"{environment}_bronze_landing_minsur.marcobre_fileserver.{new_table_name}")

# Registrar el destino para monitoreo de pipeline y depuración
print(f"Delta table successfully written to: {environment}_bronze_landing_minsur.marcobre_fileserver.{new_table_name}")